# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsiddan25/program/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Growing content is 37.6% longer and 20% younger. The label comes from the data such as analyzing the count, words, age, etc. The central hypothesis was examining how the upward trend differs from the downward trend of pages. They did notice a difference. The next steps with this info was to expand older pages (make it longer) and review aging pages.
The next finding is that content peaks at 61-90 days. The content declines after 270 days (approx 9 months). So, the suggestion for next steps is to review pages before they hit the 9-12 month.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

%pip -q install duckdb huggingface_hub scikit-learn

import os
import getpass

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): "
)


Paste your Hugging Face READ token (hf_...): ··········


In [2]:
import os
import sys
import duckdb
import numpy as np
import pandas as pd
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.13.15
pandas: 2.2.3
NumPy: 2.1.3
scikit-learn: 1.6.1


In [3]:
con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_content": (
        f"read_parquet('{REL}/dim_content.parquet')"
    ),
    "fact_daily": (
        f"read_parquet("
        f"'{REL}/fact_content_daily_performance/**/*.parquet'"
        f")"
    ),
}

print("DuckDB connection and table paths are ready.")

DuckDB connection and table paths are ready.


In [ ]:
MONTHS = [
    "2026-02",
    "2026-03",
    "2026-04",
    "2026-05",
    "2026-06",
]

monthly_queries = []

for month in MONTHS:
    monthly_queries.append(f"""
        SELECT
            client_hash_id,
            content_hash_id,
            '{month}' AS month_key,

            COUNT(*) AS observed_days,

            SUM(gsc_impressions) AS impressions,
            SUM(gsc_clicks) AS clicks,
            SUM(gsc_sum_position) AS sum_position,

            AVG(gsc_impressions) AS daily_impression_mean,
            STDDEV_SAMP(gsc_impressions) AS daily_impression_std,

            STDDEV_SAMP(
                CASE
                    WHEN gsc_impressions > 0
                    THEN gsc_avg_position
                END
            ) AS daily_position_std,

            SUM(
                CASE
                    WHEN gsc_impressions > 0 THEN 1
                    ELSE 0
                END
            ) AS days_with_impressions

        FROM read_parquet(
            '{REL}/fact_content_daily_performance/'
            'month={month}/*.parquet'
        )

        WHERE gsc_data_available IS TRUE

        GROUP BY
            client_hash_id,
            content_hash_id
    """)

monthly_union_sql = "\nUNION ALL\n".join(monthly_queries)

monthly_summary = con.sql(monthly_union_sql).df()

monthly_summary = monthly_summary.sort_values(
    ["month_key", "client_hash_id", "content_hash_id"]
).reset_index(drop=True)

print(f"Monthly summary rows: {len(monthly_summary):,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.